In [1]:
# 战备
import os
import sys
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

WORKSPACE_ROOT = Path("/vla/workspace/my_tbot")
SRC_ROOT = WORKSPACE_ROOT / "src"
MODELS_ROOT = Path("/vla/.models")
DATA_ROOT = Path("/vla/workspace/data")

### 1. 模型加载

In [2]:
from lerobot.configs.policies import PreTrainedConfig

MODEL_ID = Path("/vla/workspace/models/tbot_bp_test_saved")
device = "cuda"
QWEN3_VL_PATH = Path("/vla/workspace/models/Qwen3-VL-2B-Instruct")
COSMOS_PATH = Path("/vla/workspace/models/Cosmos-Tokenizer-CI8x8")
DA3_PATH = Path("/vla/workspace/models/DA3-LARGE-1.1")
DA3_CODE_ROOT = Path("/vla/workspace/my_tbot/third_party/Depth-Anything-3")

from lerobot.policies.BP_TBot_v2.configuration_bp_tbot import BPTBotV2Config

bp_policy_cfg = BPTBotV2Config(
    device=None,
    chunk_size=50,
    n_action_steps=50,
    n_obs_steps=1,
    max_state_dim=32,
    max_action_dim=32,
    image_delta_indices=[-15, 0, 15],
    bp_num_chunks=4,
    bp_action_chunk_size=50,
)

bp_policy_cfg.device = device
bp_policy_cfg.pretrained_path = MODEL_ID
bp_policy_cfg.qwen3_vl_variant = "qwen3_vl_28l"
bp_policy_cfg.action_expert_variant = "qwen3_28l"
bp_policy_cfg.qwen3_vl_pretrained_path = str(QWEN3_VL_PATH)
bp_policy_cfg.cosmos_tokenizer_path_or_name = str(COSMOS_PATH)
bp_policy_cfg.enable_3d_queries = True
bp_policy_cfg.num_3d_query_tokens = 432
bp_policy_cfg.lambda_3d = 0.01
bp_policy_cfg.da3_model_path_or_name = str(DA3_PATH)
bp_policy_cfg.da3_code_root = str(DA3_CODE_ROOT)
bp_policy_cfg.log_da3_teacher_timing = True

# tbot_bp / BPObsEncoder 配置：必须和保存 ckpt 时的结构一致。
bp_policy_cfg.bp_num_chunks = 20
bp_policy_cfg.bp_action_chunk_size = 50
bp_policy_cfg.bp_vision_model_name = "vit_base_patch16_clip_224.openai"
bp_policy_cfg.bp_vision_pretrained = False
bp_policy_cfg.bp_token_dim = 768
bp_policy_cfg.bp_image_feature_aggregation = "cls"
bp_policy_cfg.bp_share_rgb_model = True
bp_policy_cfg.bp_use_vision_norm = True
bp_policy_cfg.bp_freeze_vision_encoder = False
bp_policy_cfg.bp_use_action_step_embedding = True
bp_policy_cfg.bp_use_modality_type_embedding = True
bp_policy_cfg.bp_use_chunk_position_embedding = True

bp_policy_cfg.validate_features()
print("MODEL_ID:", MODEL_ID)
print("policy type:", bp_policy_cfg.type)


/vla/.conda/miniconda3/envs/mytbot/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


MODEL_ID: /vla/workspace/models/tbot_bp_test_saved
policy type: tbot_bp


In [3]:
from lerobot.policies.BP_TBot_v2.modeling_bp_tbot import TBotBPPolicy

policy = TBotBPPolicy.from_pretrained(MODEL_ID, config=bp_policy_cfg, strict=False).to(device).eval()
print(policy.name)
print(policy.model.bp_obs_encoder)
print("bp_obs_encoder in state_dict:", any(k.startswith("model.bp_obs_encoder.") for k in policy.state_dict()))
# 加载保存好的 /vla/workspace/models/tbot_bp_test_saved


[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
[INFO ] using MLP layer as FFN
Loading weights from local directory
Loading weights from local directory


tbot_bp
BPObsEncoder(
  (chunk_encoder): BPTransformerObsEncoder(
    (key_model_map): ModuleDict(
      (image_0): VisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
          (norm): Identity()
        )
        (pos_drop): Dropout(p=0.0, inplace=False)
        (patch_drop): Identity()
        (norm_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (blocks): Sequential(
          (0): Block(
            (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): Attention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (q_norm): Identity()
              (k_norm): Identity()
              (attn_drop): Dropout(p=0.0, inplace=False)
              (norm): Identity()
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
           

## 2. 数据集加载

In [5]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
# 必须对齐训练时的代码，即需要有timestamps

delta_timestamps = {
    "action": [i / 30 for i in range(50)],
    "observation.images.cam_high": [-0.5, 0.0, 0.5],
    "observation.images.cam_left_wrist": [-0.5, 0.0, 0.5],
    "observation.images.cam_right_wrist": [-0.5, 0.0, 0.5],
}

da_current = LeRobotDataset('/vla/workspace/data/robotwin2.0/adjust_bottle/aloha-agilex_clean_50',delta_timestamps=delta_timestamps)
da_prompt = LeRobotDataset('/vla/workspace/data/robotwin2.0/adjust_bottle/aloha-agilex_clean_50')
from lerobot.datasets.behavior_prompt_dataset import BehaviorPromptLeRobotDataset,BehaviorPromptConfig
config = BehaviorPromptConfig(prompt_action_chunk_size=50, 
    max_prompt_chunks=None, same_episode_policy='avoid', 
    seed=0, num_chunks=20, height=224, width=224, max_state_dim=32, max_action_dim=32, 
    qwen3_vl_processor_path=str(QWEN3_VL_PATH), action_mode='delta'
)

bp_ds = BehaviorPromptLeRobotDataset.with_default_transforms_v2(da_current, da_prompt, config)

Hydrating transform InjectMissingStateActionTransformFn (robot_type=aloha, resolved=aloha, action_seq_len=1, state_seq_len=1, placeholder_dim=14)
Hydrating transform NormalizeTransformFn with dataset.meta.stats (robot_type=aloha, resolved=aloha) and selected_keys (selected_keys=['observation.state', 'action'])
Hydrating transform ComposeFieldsTransform with mapping (robot_type=aloha, resolved=aloha)
Hydrating transform DeltaActionTransformFn with mapping and mask (robot_type=aloha, resolved=aloha)
Hydrating transform RemapImageKeyTransformFn with mapping (robot_type=aloha, resolved=aloha)


In [6]:
for i, step in enumerate(bp_ds.transform.transforms):
    print(f"data process step:  [{i}] {step.__class__.__name__}")

data process step:  [0] BPRemapImageKeyTransformFn
data process step:  [1] BPPadOrSampleChunksFn
data process step:  [2] BPResizeImagesWithPadFn
data process step:  [3] BPComposeFieldsTransform
data process step:  [4] BPDeltaActionTransformFn
data process step:  [5] BPNormalizeTransformFn
data process step:  [6] BPPadStateAndActionTransformFn
data process step:  [7] InjectMissingStateActionTransformFn
data process step:  [8] DeltaActionTransformFn
data process step:  [9] ResizeImagesWithPadFn
data process step:  [10] RemapImageKeyTransformFn
data process step:  [11] NormalizeTransformFn
data process step:  [12] ComposeFieldsTransform
data process step:  [13] PadStateAndActionTransformFn
data process step:  [14] ImgOnlyQwen3VLTransformFn


## 3. 推理与训练

In [7]:
from torch.utils.data import DataLoader
from torch.utils.data._utils.collate import default_collate
# 用bp_ds[0],bp_ds[1] 构建batch size =2 的batch
samples = [bp_ds[0],bp_ds[1]]
batch = default_collate(samples)

In [8]:
import torch
def move_to_device(x, device):
    if isinstance(x, torch.Tensor):
        return x.to(device)
    if isinstance(x, dict):
        return {k: move_to_device(v, device) for k, v in x.items()}
    if isinstance(x, list):
        return [move_to_device(v, device) for v in x]
    if isinstance(x, tuple):
        return tuple(move_to_device(v, device) for v in x)
    return x
batch_for_forward = move_to_device(batch, device)

### 训练

In [9]:
import torch
policy.train()
with torch.no_grad():
    loss, loss_dict_reloaded = policy.forward(batch_for_forward)
print(loss)
for key, value in loss_dict_reloaded.items():
    if not key.startswith("loss_action_dim"):
        print(key, value)
loss


[INFO ] Selecting reference view using strategy: saddle_balanced
tensor(0.1803, device='cuda:0')
loss 0.18028753995895386
loss_action 0.1630660593509674
loss_gen 1.4121415615081787
loss_3d 0.31000733375549316
time_3d_teacher_forward_s 0.2487260103225708
loss_3d_q13_t11 0.09938459098339081
loss_3d_q19_t15 0.1924128532409668
loss_3d_q23_t19 0.3963194787502289
loss_3d_q27_t23 0.5519124269485474


tensor(0.1803, device='cuda:0')

### 推理

In [10]:
policy.eval()
actions, _ = policy.predict_action_chunk(batch_for_forward)
actions


tensor([[[ 8.3810e-02, -1.4398e-01, -1.1677e-01,  ...,  4.1144e-03,
          -4.3572e-03,  4.4104e-03],
         [ 8.4118e-02, -1.4162e-01, -1.1454e-01,  ...,  3.1621e-03,
          -3.9671e-03,  5.6346e-04],
         [ 8.7317e-02, -1.4524e-01, -1.1613e-01,  ...,  2.0637e-03,
          -5.2630e-03,  2.0078e-03],
         ...,
         [-3.1441e+00,  1.6625e+00,  8.7823e-01,  ..., -6.2889e-03,
          -2.9784e-03, -7.9483e-03],
         [-3.1716e+00,  1.6586e+00,  8.7560e-01,  ..., -7.4465e-03,
          -4.2334e-03, -7.9787e-03],
         [-3.1837e+00,  1.6520e+00,  8.6481e-01,  ..., -8.1145e-03,
          -4.4485e-03, -8.2532e-03]],

        [[ 8.3381e-02, -1.4453e-01, -1.1298e-01,  ...,  2.2169e-03,
          -8.3226e-03,  3.4561e-03],
         [ 8.7200e-02, -1.3811e-01, -1.1478e-01,  ...,  5.4042e-03,
          -7.5851e-03,  2.5646e-03],
         [ 8.6028e-02, -1.3620e-01, -1.1199e-01,  ...,  2.0877e-03,
          -6.9454e-03, -3.8039e-03],
         ...,
         [ 7.2719e-02, -1